<a href="https://colab.research.google.com/github/AgathaCRuiz/Voice-Powered-Labs/blob/main/01_base_whisper_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
language = 'pt'

1. Gravação de Aúdio com Python (e uma pitada de JavaScript)

In [ ]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d56df48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)' % (sec * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

print('Ouvindo...\n')
record_file = record()
display(Audio(record_file, autoplay=True))

Ouvindo...



<IPython.core.display.Javascript object>

2. Reconhecimento de fala com Whisper (OpenAi)

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import whisper

model = whisper.load_model("small")
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Em poucas palavras, o que é programação.


3. Integração com API do Chagpt

In [ ]:
!pip install openai

import openai

openai.api_key = 'sk-...' # Sua chave vai aqui

response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=[ { "role": "user", "content": transcription } ]
)

chatgpt_response = response.choices[0].message.content
print(chatgpt_response)

3.1. Integração com API do Chagpt

In [ ]:
!pip uninstall -y google-generativeai google-genai
!pip install -q -U google-genai

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
Found existing installation: google-genai 1.68.0
Uninstalling google-genai-1.68.0:
  Successfully uninstalled google-genai-1.68.0


In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

meu_modelo = "gemini-2.5-flash"

try:
    response = client.models.generate_content(
        model=meu_modelo,
        contents=transcription
    )

    print(response.text)

except Exception as e:
    print(f"Erro ao processar com Gemini 2.5: {e}")

É **dar instruções a um computador**, usando uma linguagem específica (código), para que ele execute tarefas ou resolva problemas.


4. Sintetizando a Resposta de ChatGPT Como Voz (gTTS)

In [ ]:
!pip install gTTS

In [ ]:
import re # Biblioteca para limpeza de texto
from gtts import gTTS
from IPython.display import Audio, display

# 1. Limpando o texto (Removendo os asteriscos do Markdown)
# O código abaixo remove todos os '**' do texto antes de ler
texto_limpo = re.sub(r'\*', '', response.text)

# 2. Configuração da síntese com o texto limpo
language = 'pt-br'
gtts_object = gTTS(text=texto_limpo, lang=language, slow=False)

# 3. Salvando o arquivo de áudio
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# 4. Reproduzindo o áudio
print("Sintetizando voz...")
display(Audio(response_audio, autoplay=True))

Sintetizando voz...
